# 1. 生成保留需要字段的CSV原始数据文件

In [21]:
# 保留字段 id,real_time,sip_str,dip_str,sport,dport,class,rule,signature_priority,category_label,proto
import pandas as pd

file_name_list = [
    'monday-alert-label.csv',
   'tuesday-alert-label.csv',
   'wednesday-alert-label.csv',
   'thursday-alert-label.csv',
   'friday-enhance-portscan-alert-label.csv'
]
for file_name in file_name_list:
    file_path = f'../../alert_label_merge_csv_correct/{file_name}'
    file_path = f'../../alert_label_merge_csv_correct/{file_name}'
    # 读取CSV文件
    df = pd.read_csv(file_path)
    # 过滤src_addr,dst_addr为空的数据
    filtered_df = df[
        df['src_addr'].notna() & df['dst_addr'].notna() & 
        (df['src_addr'].str.strip() != '') & (df['dst_addr'].str.strip() != '')
    ]
    # 协议netbios-ssn和ftp-data分别映射为netbios和ftp
    filtered_df['service'] = filtered_df['service'].apply(
        lambda x: 'netbios' if x=='netbios-ssn' else x
    )
    filtered_df['service'] = filtered_df['service'].apply(
        lambda x: 'ftp' if x=='ftp-data' else x
    )
    # 映射协议,将service中不为unknown的填充至proto中
    filtered_df['proto'] = filtered_df.apply(
        lambda row: row['service'] if row['service'] != 'unknown' else row['proto'],
        axis=1
    )
    # 重置索引
    filtered_df.reset_index(drop=True, inplace=True)
    # 索引作为id列
    filtered_df['id'] = filtered_df.index
    # print(filtered_df['proto'].value_counts())
    # 保留字段id,real_time,src_addr,dst_addr,src_port,dst_port,class,rule,priority,Label,proto
    filtered_df = filtered_df[['id','real_time','src_addr','dst_addr','src_port','dst_port','proto','class','rule','priority',  'Label']]

    out_path = f"../..//dataset/CIC-IDS2017-format-data/correct_data/{file_name}"
    filtered_df.to_csv(out_path, index=False)

In [ ]:
# 统计service

file_name_list = [
    'monday-alert-label.csv',
   'tuesday-alert-label.csv',
   'wednesday-alert-label.csv',
   'thursday-alert-label.csv',
   'friday-enhance-portscan-alert-label.csv'
]
merge_df = pd.DataFrame()
for file_name in file_name_list:
    # 读取CSV文件
    file_path = f'../../alert_label_merge_csv/{file_name}'
    df = pd.read_csv(file_path)
    # 合并
    merge_df = pd.concat([merge_df, df], ignore_index=True)
merge_df['service'].value_counts()   

# 2. 产生训练集，测试集合
  - 训练集：周一所有误报流量+周二~周五一半的误报告警+周二~周五每个攻击类别一半告警
  - 测试集：周二~周五的剩余告警
  - 因snort3能力限制，对几乎未检出的周二上午、周三下午涉及的FTP-Patator、Heartbleed攻击类型不进行评估

In [1]:
import pandas as pd
# 周一~周五数据读入
monday_df = pd.read_csv('../..//dataset/CIC-IDS2017-format-data/correct_data/monday-alert-label.csv')
tuesday_df = pd.read_csv('../..//dataset/CIC-IDS2017-format-data/correct_data/tuesday-alert-label.csv')
wednesday_df = pd.read_csv('../..//dataset/CIC-IDS2017-format-data/correct_data/wednesday-alert-label.csv')
thursday_df = pd.read_csv('../..//dataset/CIC-IDS2017-format-data/correct_data/thursday-alert-label.csv')
friday_df = pd.read_csv('../..//dataset/CIC-IDS2017-format-data/correct_data/friday-enhance-portscan-alert-label.csv')

# 合并周一~周五数据
merged_df = pd.concat([monday_df, tuesday_df, wednesday_df, thursday_df, friday_df], ignore_index=True)
# 按照real_time时间进行排序
merged_df = merged_df.sort_values(by='real_time')
# 重置索引
merged_df = merged_df.reset_index(drop=True)
# 重新编码id
merged_df['id'] = range(0, len(merged_df))

In [5]:
merged_df.to_csv("../../", index=False)

In [2]:
merged_df['Label'].value_counts()

DoS Hulk                     349325
BENIGN                       195276
DDoS                         115562
DoS GoldenEye                 12749
DoS Slowhttptest               3123
SSH-Patator                    2980
DoS slowloris                  2315
Web Attack ?Brute Force        1823
Web Attack ?XSS                1276
Bot                             566
PortScan                        527
Web Attack ?Sql Injection        19
Infiltration                     17
FTP-Patator                       3
Name: Label, dtype: int64

In [24]:
# 按照Label，每个独立的Label选取一般数据作为训练集，剩余作为测试集
unique_labels = merged_df['Label'].unique()
# 创建训练和测试集合
train_df = pd.DataFrame(columns=merged_df.columns)
test_df = pd.DataFrame(columns=merged_df.columns)

for label in unique_labels:
    label_data = merged_df[merged_df['Label'] == label]
    label_data = label_data.sort_values(by='real_time')
    label_data.reset_index(drop=True, inplace=True)
    # 选取时按照时间选择前0.7作为训练集，后0.3作为测试集
    train_size = int(len(label_data) * 0.7)
    train_data = label_data.iloc[:train_size]
    test_data = label_data.iloc[train_size:]
    # 训练集合加入
    train_df = pd.concat([train_df, train_data])
    # 测试集合加入
    test_df = pd.concat([test_df, test_data])

# 按照时间排序
train_df = train_df.sort_values(by='real_time')
test_df = test_df.sort_values(by='real_time')
# 重置索引
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
# 重置id，从0开始
train_df['id'] = range(0, len(train_df))
test_df['id'] = range(0, len(test_df))

In [25]:
# 按照label统计下分布
print("训练集和Label分布情况")
print(train_df['Label'].value_counts())
print("测试集和Label分布情况")
print(test_df['Label'].value_counts())

训练集和Label分布情况
DoS Hulk                     244527
BENIGN                       136693
DDoS                          80893
DoS GoldenEye                  8924
DoS Slowhttptest               2186
SSH-Patator                    2086
DoS slowloris                  1620
Web Attack ?Brute Force        1276
Web Attack ?XSS                 893
Bot                             396
PortScan                        368
Web Attack ?Sql Injection        13
Infiltration                     11
FTP-Patator                       2
Name: Label, dtype: int64
测试集和Label分布情况
DoS Hulk                     104798
BENIGN                        58583
DDoS                          34669
DoS GoldenEye                  3825
DoS Slowhttptest                937
SSH-Patator                     894
DoS slowloris                   695
Web Attack ?Brute Force         547
Web Attack ?XSS                 383
Bot                             170
PortScan                        159
Infiltration                      6
Web Attack

In [26]:
# 读出数据
train_df.to_csv('../..//dataset/CIC-IDS2017-format-data/correct_data/train.csv', index=False)
test_df.to_csv('../..//dataset/CIC-IDS2017-format-data/correct_data/test.csv', index=False)